# Regularised Linear Regression — Ames Housing
### Ridge · Lasso · Elastic Net · VIF · R² · Adjusted R² · AIC

This is a rebuilt version of your project notebook. Everything you had is still here, but the
broken/incorrect parts are fixed and the missing topics are added.

**What was wrong in the original notebook and what changed**

| # | Issue in your notebook | Fix here |
|---|---|---|
| 1 | `variance_inflation_factor` was called on `X_train.values` with **no constant column**. Without an intercept the formula measures "variance around 0", not around the mean, so every VIF came out wildly inflated. | VIF is computed as `1/(1-R²ⱼ)` from an auxiliary regression **with an intercept**. |
| 2 | All features with VIF > 10 were dropped **in one shot**. VIF changes after every removal — dropping them together throws away far more than necessary. | Iterative removal: drop the single worst feature, recompute, repeat. |
| 3 | VIF was run on all 275 columns including 236 one-hot dummies. Dummies from the same parent category are collinear *by construction*. | VIF is screened on the ~39 continuous/count features only. |
| 4 | `elastic_model` was never defined (cell `l1` was a stray typo), so `GridSearchCV` would raise `NameError`. | Proper `ElasticNet` instance + grid. |
| 5 | The scaler was fit on the *whole* training set, then `GridSearchCV` cross-validated on top of it → mild leakage into every fold. | `Pipeline(StandardScaler → model)` inside `GridSearchCV`, so scaling is refit inside each fold. |
| 6 | `alpha` grids started at 0.001. `SalePrice` is in dollars (~180,000), so tiny alphas mean *no* regularisation and Lasso/ElasticNet take minutes to converge. | Alpha grids on a log scale in a range that actually matters for this target. |
| 7 | Ridge alphas were tuned by looking at **test RMSE** — that is tuning on the test set. | Alphas tuned by 5-fold CV on the training set; the test set is touched exactly once, at the end. |
| 8 | `test_size=0.2` while the task text said 10%. | `test_size=0.1`, `random_state=101`. |
| 9 | No Lasso, no R², no adjusted R², no AIC. | All added, with the maths written out. |
| 10 | 236 columns are `bool` dtype — statsmodels and some sklearn paths choke on that. | Cast to `int` up front. |

**Nothing needs to change in your CSV.** It is clean: 2919 rows, 276 columns, zero nulls, no duplicate
rows, no constant columns. One thing to be aware of — it already contains engineered features
(`total_sf`, `total_bath`, `House Age`, `Remod Age`) that are *exact arithmetic functions* of other
columns. That produces perfect multicollinearity, which is exactly what the VIF section will catch.

## 1. Imports and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
RANDOM_STATE = 101

In [ ]:
# change the path if you are on Colab, e.g. "/content/final_cleaned_df (3) (2).csv"
df = pd.read_csv("final_cleaned_df__3___2_.csv")

# FIX: 236 columns are stored as bool. Cast to int so statsmodels / numpy maths behave.
bool_cols = df.select_dtypes("bool").columns
df[bool_cols] = df[bool_cols].astype(int)

print("shape:", df.shape)
print("missing values:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
df.head()

In [ ]:
df["SalePrice"].describe()

### A quick look at the target

`SalePrice` is right-skewed. Linear regression assumes roughly constant-variance, symmetric errors,
so this matters. We will model the raw dollar value first (so the metrics are interpretable), and
revisit the log transform at the end as a bonus.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["SalePrice"], bins=50, kde=True, ax=ax[0])
ax[0].set_title(f"SalePrice  (skew = {df['SalePrice'].skew():.2f})")
sns.histplot(np.log(df["SalePrice"]), bins=50, kde=True, ax=ax[1], color="darkorange")
ax[1].set_title(f"log(SalePrice)  (skew = {np.log(df['SalePrice']).skew():.2f})")
plt.tight_layout(); plt.show()

## 2. Train / test split

The test set is locked away now and used **once**, at the very end. Every tuning decision
(alpha, l1_ratio, which features to drop) is made using cross-validation on the training set only.
This is the single most important habit in the whole notebook.

In [ ]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE
)
print("train:", X_train.shape, " test:", X_test.shape)

### Why scaling is mandatory here

Ridge, Lasso and Elastic Net all penalise the **size** of the coefficients. A feature measured in
square feet (0–5000) and a 0/1 dummy are on completely different scales, so without standardisation
the penalty would punish them unequally and arbitrarily. OLS does not care about scaling (it is
equivariant); regularised models care enormously.

We use `StandardScaler` inside a `Pipeline` so that it is refit on the training portion of **each
CV fold** rather than on the whole training set. Fitting the scaler once outside CV leaks the mean
and standard deviation of the validation fold into training.

## 3. The metric toolkit: MAE, RMSE, R², Adjusted R², AIC, BIC

Four of these you asked about specifically, so here is the maths.

**R²** — the fraction of variance in $y$ explained by the model.

$$R^2 = 1 - \frac{RSS}{TSS} = 1 - \frac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}$$

Problem: adding *any* predictor, even pure noise, can never decrease training R². So R² on the
training set always rewards bigger models.

**Adjusted R²** — R² with a penalty for the number of predictors $k$ on $n$ observations.

$$R^2_{adj} = 1 - (1-R^2)\,\frac{n-1}{n-k-1}$$

Now a useless predictor *lowers* the score, because it costs a degree of freedom without reducing
RSS enough to pay for it. Use it to compare models fit on the **same training data**; it is
meaningless as a test-set metric (the "number of predictors" penalty has no role out of sample).

**AIC** — Akaike Information Criterion. Estimates out-of-sample prediction error from in-sample fit
plus a complexity penalty. Lower is better.

$$AIC = -2\ln \hat{L} + 2p$$

For OLS with Gaussian errors the log-likelihood has a closed form:

$$\ln \hat L = -\frac{n}{2}\Big[\ln(2\pi) + \ln\!\big(\tfrac{RSS}{n}\big) + 1\Big]$$

where $p = k + 2$ (the $k$ slopes, the intercept, and the error variance $\sigma^2$).

**BIC** replaces the $2p$ penalty with $p\ln n$ — harsher on complexity for any $n > 7$, so it
prefers smaller models than AIC does.

Three things people get wrong about AIC:

1. The absolute number is meaningless. Only *differences* between models fit to the **same
   response values** matter. An AIC of 59,782 is not "bad" — it is just a number.
2. A difference of roughly 2 is negligible; > 10 is decisive.
3. For Ridge and Lasso, $k$ is not simply "how many columns I passed in", because the penalty
   shrinks coefficients and costs you less than a full degree of freedom. We handle that below.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def adjusted_r2(y_true, y_pred, k):
    '''R^2 adjusted for k predictors. Only meaningful on the data the model was fit to.'''
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)

def aic_bic(y_true, y_pred, k):
    '''
    AIC / BIC for a Gaussian linear model, from the residual sum of squares.
    k = effective number of predictors (slopes). p = k + intercept + sigma^2.
    '''
    n = len(y_true)
    rss = np.sum((np.asarray(y_true) - np.asarray(y_pred)) ** 2)
    log_lik = -n / 2 * (np.log(2 * np.pi) + np.log(rss / n) + 1)
    p = k + 2
    return -2 * log_lik + 2 * p, -2 * log_lik + np.log(n) * p

### Effective degrees of freedom for Ridge and Lasso

This is the subtle bit that most tutorials skip.

* **OLS**: $df = k$, the number of predictors.
* **Lasso**: $df \approx$ the number of **non-zero** coefficients. This is a genuine theoretical
  result (Zou, Hastie & Tibshirani, 2007), not a hand-wave — and it is what makes Lasso so
  convenient for AIC-based model comparison.
* **Ridge**: nothing is ever exactly zero, but every coefficient is shrunk, so the model uses
  *less* than $k$ degrees of freedom. Using the singular values $d_j$ of the scaled design matrix,

  $$df(\alpha) = \sum_j \frac{d_j^2}{d_j^2 + \alpha}$$

  At $\alpha = 0$ this equals $k$; as $\alpha \to \infty$ it goes to 0.

In [ ]:
def ridge_effective_df(X_scaled, alpha):
    d = np.linalg.svd(X_scaled, compute_uv=False)
    return np.sum(d**2 / (d**2 + alpha))

def report(name, model, X_tr, X_te, k, store):
    '''Fit-free evaluator: model is already fitted. Returns one row of the comparison table.'''
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    aic, bic = aic_bic(y_train, pred_tr, k)
    row = {
        "Model": name,
        "Eff. k": round(k, 1),
        "Train R2": r2_score(y_train, pred_tr),
        "Train Adj R2": adjusted_r2(y_train, pred_tr, k),
        "AIC": aic,
        "BIC": bic,
        "Test R2": r2_score(y_test, pred_te),
        "Test MAE": mean_absolute_error(y_test, pred_te),
        "Test RMSE": rmse(y_test, pred_te),
    }
    store.append(row)
    print(f"{name}")
    print(f"  train R2 {row['Train R2']:.4f} | adj R2 {row['Train Adj R2']:.4f} | "
          f"AIC {aic:,.0f} | BIC {bic:,.0f} | eff. k {k:.1f}")
    print(f"  TEST  R2 {row['Test R2']:.4f} | MAE ${row['Test MAE']:,.0f} | RMSE ${row['Test RMSE']:,.0f}")
    return row

results = []

## 4. Baseline: plain OLS on all 275 features

In [ ]:
ols_pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ols_pipe.fit(X_train, y_train)

k_ols = int(np.sum(np.abs(ols_pipe.named_steps["model"].coef_) > 1e-8))
report("OLS — all features", ols_pipe, X_train, X_test, k_ols, results);

Read that output carefully:

* Train R² is much higher than test R² — the gap is the overfitting we are about to attack.
* Adjusted R² sits below train R², and the size of that drop tells you how much of the fit you are
  buying with sheer parameter count rather than signal.
* The AIC number is the reference point. Every model below is compared against **this** value.

## 5. Multicollinearity and VIF

**What multicollinearity does.** If two predictors carry nearly the same information, OLS cannot
decide how to split the credit between them. The fitted plane is still fine — *predictions barely
suffer* — but the individual coefficients become unstable: huge standard errors, signs that flip
when you resample the data, and coefficients that are impossible to interpret. This is why VIF is
an **interpretation** tool, not a prediction tool.

**Variance Inflation Factor.** For feature $j$, regress it on *all the other features* and take the
$R^2_j$ of that auxiliary regression:

$$VIF_j = \frac{1}{1 - R_j^2}$$

It answers: *by what factor is the variance of $\hat\beta_j$ inflated, compared to a world where
feature $j$ were uncorrelated with everything else?* A VIF of 9 means the standard error of that
coefficient is 3× larger ($\sqrt{9}$) than it needed to be.

| VIF | Reading |
|---|---|
| ≈ 1 | independent of the rest |
| 1 – 5 | fine |
| 5 – 10 | worth inspecting |
| > 10 | serious; the coefficient is not trustworthy |
| ∞ | **exact** linear dependence — one column is an arithmetic combination of others |

A heatmap or a correlation matrix only shows you *pairwise* relationships. VIF catches the case
where feature A is perfectly predicted by B + C + D even though it correlates weakly with each one
individually. That is the whole reason VIF exists.

**Two corrections to the original notebook:**

1. `statsmodels.variance_inflation_factor(X.values, i)` assumes your matrix already contains a
   constant column. Pass raw `X_train.values` and every auxiliary regression is forced through the
   origin, so $R^2_j$ is computed around 0 instead of around the mean and every VIF is inflated
   nonsense. Either `sm.add_constant(X)` first, or compute VIF yourself as below — the function
   here uses `LinearRegression`, which fits an intercept by default, so it is correct by
   construction.
2. Screen only the continuous / count features. One-hot dummies from the same parent variable are
   collinear by design (knowing 14 of 15 `Neighborhood` dummies tells you the 15th), so their VIFs
   are structurally high and mean nothing. Dropping them would destroy the category.

In [ ]:
def vif_table(frame):
    '''VIF_j = 1 / (1 - R2_j) from an auxiliary regression WITH an intercept.'''
    Z = StandardScaler().fit_transform(frame)   # scaling only for numerical stability
    vifs = []
    for i in range(Z.shape[1]):
        others = np.delete(Z, i, axis=1)
        r2 = LinearRegression().fit(others, Z[:, i]).score(others, Z[:, i])
        vifs.append(np.inf if r2 > 1 - 1e-6 else 1 / (1 - r2))
    return (pd.DataFrame({"Feature": frame.columns, "VIF": vifs})
              .sort_values("VIF", ascending=False)
              .reset_index(drop=True))

# continuous / count features only — anything with more than 2 distinct values
continuous = [c for c in X_train.columns if X_train[c].nunique() > 2]
print(f"{len(continuous)} continuous features screened, "
      f"{X_train.shape[1] - len(continuous)} dummies left alone")

vif_df = vif_table(X_train[continuous])
vif_df.head(20)

### Those infinities are not a bug — read them

Several features come back as `inf`, meaning $R^2_j = 1$: some column is an **exact** linear
combination of others. Look at what is in the data:

* `total_sf`  = `Total Bsmt SF` + `1st Flr SF` + `2nd Flr SF` (an engineered feature)
* `Total Bsmt SF` = `BsmtFin SF 1` + `BsmtFin SF 2` + `Bsmt Unf SF`
* `Gr Liv Area` = `1st Flr SF` + `2nd Flr SF` + `Low Qual Fin SF`
* `total_bath` = `Full Bath` + `0.5·Half Bath` + `Bsmt Full Bath` + `0.5·Bsmt Half Bath`
* `House Age` = `Yr Sold` − `Year Built`, and `Remod Age` = `Yr Sold` − `Year Remod/Add`

Whoever cleaned this file added the engineered totals **without removing the parts**. That is the
single biggest structural issue in the dataset, and it is exactly what VIF is for. Verify it:

In [ ]:
checks = {
    "total_sf == TotalBsmt + 1stFlr + 2ndFlr":
        np.allclose(X_train["total_sf"], X_train["Total Bsmt SF"] + X_train["1st Flr SF"] + X_train["2nd Flr SF"]),
    "Total Bsmt SF == Fin1 + Fin2 + Unf":
        np.allclose(X_train["Total Bsmt SF"], X_train["BsmtFin SF 1"] + X_train["BsmtFin SF 2"] + X_train["Bsmt Unf SF"]),
    "Gr Liv Area == 1st + 2nd + LowQual":
        np.allclose(X_train["Gr Liv Area"], X_train["1st Flr SF"] + X_train["2nd Flr SF"] + X_train["Low Qual Fin SF"]),
    "House Age == Yr Sold - Year Built":
        np.allclose(X_train["House Age"], X_train["Yr Sold"] - X_train["Year Built"]),
    "Remod Age == Yr Sold - Year Remod/Add":
        np.allclose(X_train["Remod Age"], X_train["Yr Sold"] - X_train["Year Remod/Add"]),
}
for k_, v_ in checks.items():
    print(f"{v_}  <-  {k_}")

### Iterative removal (the fix for issue #2)

Dropping every VIF > 10 at once is wrong because VIF is a *joint* quantity: remove one member of a
collinear group and the VIFs of the others collapse. The correct procedure is a loop —

1. compute all VIFs,
2. drop the single worst offender,
3. recompute,
4. stop when everything is under the threshold.

Where you have a choice within a group, keep the feature that is easiest to interpret or most
directly measured.

In [ ]:
def iterative_vif_filter(frame, threshold=10.0, verbose=True):
    keep = list(frame.columns)
    dropped = []
    while True:
        t = vif_table(frame[keep])
        worst = t.iloc[0]
        if worst["VIF"] <= threshold:
            break
        dropped.append((worst["Feature"], worst["VIF"]))
        if verbose:
            v = worst["VIF"]
            print(f"drop  {worst['Feature']:<20} VIF = {'inf' if np.isinf(v) else f'{v:8.2f}'}")
        keep.remove(worst["Feature"])
    return keep, dropped, vif_table(frame[keep])

kept_continuous, dropped_list, final_vif = iterative_vif_filter(X_train[continuous], threshold=10.0)
print(f"\nremoved {len(dropped_list)} of {len(continuous)} continuous features")

In [ ]:
final_vif

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(data=final_vif, y="Feature", x="VIF", ax=ax, color="steelblue")
ax.axvline(5, color="orange", ls="--", label="inspect (5)")
ax.axvline(10, color="red", ls="--", label="serious (10)")
ax.set_title("VIF after iterative removal — every feature is now below 10")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
vif_dropped = [f for f, _ in dropped_list]
X_train_vif = X_train.drop(columns=vif_dropped)
X_test_vif  = X_test.drop(columns=vif_dropped)
print("dropped:", vif_dropped)
print(X_train_vif.shape, X_test_vif.shape)

ols_vif = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ols_vif.fit(X_train_vif, y_train)
k_vif = int(np.sum(np.abs(ols_vif.named_steps["model"].coef_) > 1e-8))
report("OLS — VIF filtered", ols_vif, X_train_vif, X_test_vif, k_vif, results);

**What to notice.** Test RMSE barely moves — removing collinear features did **not** improve
prediction, and it was never going to. But AIC improves slightly (same fit, fewer parameters) and,
far more importantly, the surviving coefficients are now interpretable. If your project report has
a "which features drive price" section, it must be built on this model, not the 275-feature one.

This is the honest summary of VIF: **an inference tool, not an accuracy tool.**

## 6. Why regularisation

OLS minimises RSS and nothing else. With 275 features and 2,627 training rows it has enough freedom
to chase noise, which is the train/test R² gap you saw. Regularisation adds a penalty on coefficient
size, deliberately accepting a little **bias** in exchange for a large drop in **variance**.

$$\text{Ridge:}\quad \min_\beta \; \|y - X\beta\|^2 + \alpha\sum_j \beta_j^2 \qquad (L_2)$$

$$\text{Lasso:}\quad \min_\beta \; \|y - X\beta\|^2 + \alpha\sum_j |\beta_j| \qquad (L_1)$$

$$\text{Elastic Net:}\quad \min_\beta \; \|y - X\beta\|^2 + \alpha\Big[\rho\sum_j|\beta_j| + \tfrac{1-\rho}{2}\sum_j\beta_j^2\Big]$$

| | Ridge (L2) | Lasso (L1) | Elastic Net |
|---|---|---|---|
| Shrinks coefficients | yes | yes | yes |
| Sets them exactly to 0 | **no** | **yes** | yes |
| With a group of correlated features | splits the weight across all of them | picks ~one, zeroes the rest (arbitrarily) | keeps the group together |
| Can handle k > n | yes | yes (selects at most n) | yes |
| Use when | everything matters a bit; heavy collinearity | you want a sparse, readable model | correlated features *and* you want sparsity |

The geometric reason Lasso zeroes things: the L1 constraint region is a diamond with corners **on
the axes**, so the RSS contours tend to touch it at a corner where some coefficients are exactly 0.
The L2 region is a sphere — no corners, no exact zeros.

`alpha` controls the strength: `alpha=0` is plain OLS, `alpha→∞` drives every coefficient to 0
(prediction becomes the mean of y). `l1_ratio` = $\rho$ mixes the two penalties: 0 is pure Ridge,
1 is pure Lasso.

We keep **all 275 features** from here on. Regularisation is the principled alternative to VIF-based
deletion: instead of throwing correlated columns away, it shrinks them. That is the whole point.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 7. Ridge Regression

In [ ]:
ridge_grid = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("model", Ridge(max_iter=10000))]),
    param_grid={"model__alpha": np.logspace(0, 4, 30)},
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=-1,
)
ridge_grid.fit(X_train, y_train)

best_alpha_ridge = ridge_grid.best_params_["model__alpha"]
print(f"best alpha: {best_alpha_ridge:.2f}   (CV RMSE ${-ridge_grid.best_score_:,.0f})")

In [ ]:
cvres = pd.DataFrame(ridge_grid.cv_results_)
plt.figure(figsize=(7, 4))
plt.plot(cvres["param_model__alpha"].astype(float), -cvres["mean_test_score"], marker="o", ms=3)
plt.axvline(best_alpha_ridge, color="red", ls="--", label=f"best alpha = {best_alpha_ridge:.1f}")
plt.xscale("log"); plt.xlabel("alpha (log scale)"); plt.ylabel("5-fold CV RMSE ($)")
plt.title("Ridge: validation curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# effective degrees of freedom, not the raw feature count
X_train_scaled = StandardScaler().fit_transform(X_train)
k_ridge = ridge_effective_df(X_train_scaled, best_alpha_ridge)
print(f"raw features: {X_train.shape[1]}   effective df at alpha={best_alpha_ridge:.1f}: {k_ridge:.1f}")

report("Ridge", ridge_grid.best_estimator_, X_train, X_test, k_ridge, results);

In [ ]:
# how coefficients shrink as alpha grows
alphas = np.logspace(0, 5, 40)
paths = []
Xs = StandardScaler().fit_transform(X_train)
for a in alphas:
    paths.append(Ridge(alpha=a).fit(Xs, y_train).coef_)
paths = np.array(paths)

plt.figure(figsize=(8, 5))
for j in range(paths.shape[1]):
    plt.plot(alphas, paths[:, j], lw=0.7)
plt.xscale("log"); plt.axhline(0, color="k", lw=0.8)
plt.axvline(best_alpha_ridge, color="red", ls="--", label="chosen alpha")
plt.xlabel("alpha"); plt.ylabel("coefficient value")
plt.title("Ridge coefficient paths — everything shrinks toward 0, nothing reaches it")
plt.legend(); plt.tight_layout(); plt.show()

## 8. Lasso Regression

Same machinery, L1 penalty. The alpha range starts higher because with `SalePrice` in dollars,
`alpha=0.01` is effectively no penalty at all — that is why the original grid was so slow to
converge.

In [ ]:
lasso_grid = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=20000))]),
    param_grid={"model__alpha": np.logspace(1, 4, 30)},
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=-1,
)
lasso_grid.fit(X_train, y_train)

best_alpha_lasso = lasso_grid.best_params_["model__alpha"]
lasso_best = lasso_grid.best_estimator_.named_steps["model"]
k_lasso = int(np.sum(lasso_best.coef_ != 0))

print(f"best alpha: {best_alpha_lasso:.2f}")
print(f"features kept: {k_lasso} of {X_train.shape[1]}  "
      f"({X_train.shape[1] - k_lasso} driven to exactly zero)")

report("Lasso", lasso_grid.best_estimator_, X_train, X_test, k_lasso, results);

In [ ]:
lasso_coefs = (pd.Series(lasso_best.coef_, index=X_train.columns)
                 .loc[lambda s: s != 0]
                 .sort_values(key=abs, ascending=False))

print("Top 20 surviving features (coefficients are in $ per 1 standard deviation of the feature):")
lasso_coefs.head(20)

In [ ]:
top = lasso_coefs.head(20).sort_values()
plt.figure(figsize=(8, 7))
colors = ["firebrick" if v < 0 else "seagreen" for v in top]
plt.barh(top.index, top.values, color=colors)
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("coefficient ($ per 1 SD)")
plt.title(f"Lasso: 20 strongest of {k_lasso} surviving features")
plt.tight_layout(); plt.show()

In [ ]:
# how many features survive as alpha grows
n_nonzero = [np.sum(Lasso(alpha=a, max_iter=20000).fit(Xs, y_train).coef_ != 0)
             for a in np.logspace(1, 4.5, 25)]
plt.figure(figsize=(7, 4))
plt.plot(np.logspace(1, 4.5, 25), n_nonzero, marker="o", ms=4)
plt.axvline(best_alpha_lasso, color="red", ls="--", label="chosen alpha")
plt.xscale("log"); plt.xlabel("alpha"); plt.ylabel("non-zero coefficients")
plt.title("Lasso performs feature selection as alpha increases")
plt.legend(); plt.tight_layout(); plt.show()

## 9. Elastic Net

Elastic Net searches a **2-D** grid: `alpha` (total penalty strength) × `l1_ratio` (how much of it
is L1). If the best `l1_ratio` comes back as 1.0, the data is telling you pure Lasso is enough.

In [ ]:
elastic_model = ElasticNet(max_iter=20000, random_state=RANDOM_STATE)   # FIX: this was never defined

param_grid = {
    "model__alpha": np.logspace(1, 4, 20),
    "model__l1_ratio": [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
}

elastic_grid = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("model", elastic_model)]),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=-1, verbose=1,
)
elastic_grid.fit(X_train, y_train)

print(elastic_grid.best_params_)

In [ ]:
enet_best = elastic_grid.best_estimator_.named_steps["model"]
k_enet = int(np.sum(enet_best.coef_ != 0))
print(f"features kept: {k_enet} of {X_train.shape[1]}")

report("Elastic Net", elastic_grid.best_estimator_, X_train, X_test, k_enet, results);

In [ ]:
# heatmap of the 2-D search
pivot = (pd.DataFrame(elastic_grid.cv_results_)
           .pivot_table(index="param_model__l1_ratio",
                        columns="param_model__alpha",
                        values="mean_test_score"))
plt.figure(figsize=(12, 4))
sns.heatmap(-pivot, cmap="viridis_r", cbar_kws={"label": "CV RMSE ($)"},
            xticklabels=[f"{float(c):.0f}" for c in pivot.columns])
plt.title("Elastic Net grid search — darker is better")
plt.xlabel("alpha"); plt.ylabel("l1_ratio")
plt.tight_layout(); plt.show()

## 10. Putting every metric side by side

In [ ]:
comparison = pd.DataFrame(results).set_index("Model")
comparison.style.format({
    "Train R2": "{:.4f}", "Train Adj R2": "{:.4f}", "Test R2": "{:.4f}",
    "AIC": "{:,.0f}", "BIC": "{:,.0f}", "Test MAE": "${:,.0f}", "Test RMSE": "${:,.0f}",
}).background_gradient(subset=["Test RMSE", "Test MAE"], cmap="RdYlGn_r")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
comparison["Test RMSE"].plot(kind="bar", ax=ax[0], color="steelblue", title="Test RMSE ($) — lower better")
comparison["Test R2"].plot(kind="bar", ax=ax[1], color="seagreen", title="Test R² — higher better")
comparison["AIC"].plot(kind="bar", ax=ax[2], color="indianred", title="AIC (train) — lower better")
ax[2].set_ylim(comparison["AIC"].min() * 0.999, comparison["AIC"].max() * 1.001)
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### How to read this table — the part that belongs in your report

**The differences are small, and that is the finding.** With n ≈ 2,600 and 275 features, OLS has
enough data that it is not catastrophically overfit, so regularisation buys refinement rather than
rescue. Report that honestly; do not manufacture a story where Ridge "wins by a mile".

What each metric is actually telling you:

* **Train R² vs Test R²** — the gap is overfitting. OLS has the widest gap; the penalised models
  narrow it by giving up some training fit.
* **Adjusted R²** — reorders the models by punishing parameter count. Note that OLS can have the
  highest raw train R² and *not* the highest adjusted R².
* **AIC / BIC** — the same idea with a stricter, likelihood-based penalty. Lasso does well here
  because it achieves nearly the same RSS with roughly a third of the parameters. BIC's harsher
  penalty ($p\ln n$ vs $2p$) favours the sparse models even more strongly.
* **Test MAE vs Test RMSE** — MAE is the typical error in dollars; RMSE squares the errors first, so
  it is dominated by a handful of expensive houses the model badly misses. When RMSE and MAE
  disagree about which model is best, it is a disagreement about how much you care about those
  outliers. Say which one you chose and why.
* **Effective k** — Ridge keeps all 275 columns but uses far fewer effective degrees of freedom;
  Lasso and Elastic Net cut the column count outright.

**Which model would I hand in?** Lasso, unless the report asks purely for accuracy. It is within
noise of the others on test error while using a fraction of the features, which makes it the only
one you can actually explain to somebody.

## 11. Residual diagnostics

Metrics summarise; residual plots tell you *where* the model fails. Two checks:

* **Residuals vs fitted** should be a formless cloud centred on 0. A funnel shape means
  heteroscedasticity — errors grow with price, which is the classic symptom of modelling a
  right-skewed target on its raw scale.
* **Q-Q plot** — points should follow the diagonal. Curling at the ends means heavy tails.

In [ ]:
from scipy import stats

best_model = lasso_grid.best_estimator_
pred_test = best_model.predict(X_test)
resid = y_test - pred_test

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

ax[0].scatter(pred_test, resid, alpha=0.5, edgecolor="k", lw=0.3)
ax[0].axhline(0, color="red", ls="--")
ax[0].set_xlabel("predicted price"); ax[0].set_ylabel("residual")
ax[0].set_title("Residuals vs fitted")

stats.probplot(resid, dist="norm", plot=ax[1])
ax[1].set_title("Q-Q plot of residuals")

ax[2].scatter(y_test, pred_test, alpha=0.5, edgecolor="k", lw=0.3)
lims = [y_test.min(), y_test.max()]
ax[2].plot(lims, lims, "r--")
ax[2].set_xlabel("actual"); ax[2].set_ylabel("predicted"); ax[2].set_title("Actual vs predicted")

plt.tight_layout(); plt.show()
print(f"residual mean ${resid.mean():,.0f}  |  residual SD ${resid.std():,.0f}")

## 12. Bonus: modelling log(SalePrice)

The residual funnel above has an obvious fix. Predict $\log(\text{SalePrice})$, then exponentiate
back to dollars. This turns a multiplicative error structure ("this model is off by 10%") into an
additive one, which is what linear regression assumes.

Note the alpha range changes completely — the target is now on a scale of about 12, not 180,000, so
the penalty that matters is thousands of times smaller. This is a good illustration of why alpha
values can never be copied between problems.

In [ ]:
y_train_log = np.log(y_train)

lasso_log = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=20000))]),
    param_grid={"model__alpha": np.logspace(-4, -1, 30)},
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=-1,
)
lasso_log.fit(X_train, y_train_log)

pred_log = np.exp(lasso_log.predict(X_test))      # back to dollars
print(f"best alpha: {lasso_log.best_params_['model__alpha']:.5f}")
print(f"features kept: {np.sum(lasso_log.best_estimator_.named_steps['model'].coef_ != 0)}")
print(f"\nTEST  R2 {r2_score(y_test, pred_log):.4f} | "
      f"MAE ${mean_absolute_error(y_test, pred_log):,.0f} | RMSE ${rmse(y_test, pred_log):,.0f}")
print(f"(compare with the dollar-scale Lasso above)")

In [ ]:
resid_log = y_test - pred_log
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(pred_test, y_test - pred_test, alpha=0.5, edgecolor="k", lw=0.3)
ax[0].axhline(0, color="red", ls="--"); ax[0].set_title("Residuals — raw dollar target")
ax[1].scatter(pred_log, resid_log, alpha=0.5, edgecolor="k", lw=0.3, color="darkorange")
ax[1].axhline(0, color="red", ls="--"); ax[1].set_title("Residuals — log target")
for a in ax:
    a.set_xlabel("predicted price"); a.set_ylabel("residual"); a.set_ylim(-150000, 150000)
plt.tight_layout(); plt.show()

The funnel flattens out and every test metric improves. If your assignment allows a target
transform, this is the version to hand in — and the reason is a modelling argument about error
structure, not a lucky hyperparameter.

## 13. Summary

**Concepts, one line each**

| Concept | One-line takeaway |
|---|---|
| **R²** | Variance explained. Never decreases when you add features, so it cannot choose a model size for you. |
| **Adjusted R²** | R² penalised by $\frac{n-1}{n-k-1}$. Comparable across models on the same training data; meaningless on a test set. |
| **AIC** | $-2\ln L + 2p$. Estimated out-of-sample error from in-sample likelihood plus a complexity penalty. Only differences matter; ~2 is noise, >10 is decisive. |
| **BIC** | AIC with a $p\ln n$ penalty. Prefers smaller models. |
| **VIF** | $1/(1-R_j^2)$. Diagnoses unstable *coefficients*, not poor *predictions*. Needs an intercept; remove features one at a time; skip the dummies; ∞ means an exact linear dependency. |
| **Ridge** | L2. Shrinks all coefficients, keeps all features, handles collinearity gracefully. Effective df < k. |
| **Lasso** | L1. Sets coefficients to exactly 0 — automatic feature selection. Picks arbitrarily within a correlated group. |
| **Elastic Net** | Mix of both. Sparsity plus stable handling of correlated groups. `l1_ratio=1` ⇒ pure Lasso, `0` ⇒ pure Ridge. |

**Method habits this notebook enforces**

1. Split first, touch the test set once.
2. Scale inside a `Pipeline` so it is refit within every CV fold.
3. Tune on cross-validated training error, never on test error.
4. Choose alpha grids on a log scale, sized to your target's units.
5. Use VIF for interpretation, regularisation for prediction — they solve different problems.
6. Report training-set criteria (adjusted R², AIC, BIC) and test-set criteria (MAE, RMSE, R²)
   separately; they answer different questions.

**Possible extensions**

* `PolynomialFeatures(degree=2, interaction_only=True)` before the penalty — regularisation is what
  makes that survivable.
* `RidgeCV` / `LassoCV` / `ElasticNetCV`, which exploit warm starts and are far faster than
  `GridSearchCV` for the same search.
* Repeated K-fold to put an error bar on the model comparison, since the gaps here are small.